# Teste de tempo no Kaggle

Mede quanto tempo cada modelo leva por trecho na GPU do Kaggle, em precisão normal (fp32) e meia precisão (fp16),
usando os mesmos 500 trechos de "engineer" em 2015 do teste feito na MX330.

Antes de rodar: **Accelerator = GPU** e **Internet = On** (painel da direita), e o dataset com os trechos adicionado em *Input*.
Depois é só **Run All**. O resumo para copiar está na última célula.

In [ ]:
import glob, time
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), "Sem GPU: ligue Accelerator = GPU no painel da direita (Session options) e rode de novo."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)

arq = glob.glob("/kaggle/input/**/trechos_engineer_2015.csv", recursive=True)
assert arq, "Não achei trechos_engineer_2015.csv: adicione o dataset em 'Add Input' no painel da direita."
df = pd.read_csv(arq[0])
trechos, alvos = df["trecho"].tolist(), list(zip(df["ini"], df["fim"]))
ref = glob.glob("/kaggle/input/**/vetor_referencia_mx330.npy", recursive=True)
vetor_mx330 = np.load(ref[0]) if ref else None
print(len(trechos), "trechos carregados;", "vetor da MX330 encontrado" if ref else "sem vetor da MX330")

In [ ]:
def extrair(nome, meia_precisao, lote=64):
    """Vetor médio da palavra nos trechos + segundos por trecho (sem contar o carregamento do modelo)."""
    tok = AutoTokenizer.from_pretrained(nome)
    mod = AutoModel.from_pretrained(nome).to("cuda").eval()
    if meia_precisao:
        mod = mod.half()
    ordem = sorted(range(len(trechos)), key=lambda i: len(trechos[i]))   # tamanhos parecidos no mesmo lote

    def rodar(indices):
        enc = tok([trechos[i] for i in indices], return_offsets_mapping=True, truncation=True,
                  max_length=512, padding=True, return_tensors="pt")
        pos = enc["offset_mapping"]
        saida = mod(input_ids=enc["input_ids"].to("cuda"),
                    attention_mask=enc["attention_mask"].to("cuda")).last_hidden_state.float().cpu()
        vetores = []
        for b, i in enumerate(indices):
            a_ini, a_fim = alvos[i]
            # tokens que se sobrepõem à palavra-alvo (1 ou mais pedaços)
            idx = [t for t, (s, e) in enumerate(pos[b].tolist()) if e > s and s < a_fim and e > a_ini]
            if idx:
                vetores.append(saida[b, idx].mean(0).numpy())
        return vetores

    with torch.no_grad():
        rodar(ordem[:lote]); torch.cuda.synchronize()          # aquecimento (não entra no tempo)
        t0 = time.perf_counter()
        vetores = []
        for k in range(0, len(ordem), lote):
            vetores += rodar(ordem[k:k + lote])
        torch.cuda.synchronize()
        segundos = time.perf_counter() - t0
    v = np.mean(vetores, axis=0)
    del mod; torch.cuda.empty_cache()
    return v / np.linalg.norm(v), segundos / len(vetores), len(vetores)

In [ ]:
MODELOS = ["bert-base-cased", "roberta-base", "answerdotai/ModernBERT-base"]
resultados, vetores = [], {}
for nome in MODELOS:
    for meia in [False, True]:
        precisao = "fp16" if meia else "fp32"
        print(f"{nome} {precisao}...", end=" ", flush=True)
        try:
            v, s, n = extrair(nome, meia)
            vetores[(nome, precisao)] = v
            resultados.append({"modelo": nome, "precisão": precisao, "ms por trecho": round(s * 1000, 2), "ocorrências": n})
            print(f"{s * 1000:.2f} ms/trecho")
        except Exception as erro:
            resultados.append({"modelo": nome, "precisão": precisao, "ms por trecho": None, "ocorrências": 0, "erro": str(erro)[:200]})
            print("ERRO:", erro)
tabela = pd.DataFrame(resultados)
tabela

## fp16 dá o mesmo vetor que fp32? E o BERT daqui bate com o da MX330?

In [ ]:
for nome in MODELOS:
    a, b = vetores.get((nome, "fp32")), vetores.get((nome, "fp16"))
    if a is not None and b is not None:
        print(f"{nome}: similaridade fp32 × fp16 = {a @ b:.4f}")
if vetor_mx330 is not None and ("bert-base-cased", "fp32") in vetores:
    print(f"bert-base-cased: similaridade Kaggle fp32 × MX330 = {vetores[('bert-base-cased', 'fp32')] @ vetor_mx330:.4f}")

## Projeção do trabalho completo no Kaggle

Trechos por modelo de cada opção da PP3 (contagem real, 2008–10 agrupado, mínimo 50, teto 500).
Usa o tempo **fp16** de cada modelo (se a similaridade fp32 × fp16 acima for ~1,0, a meia precisão não muda o resultado).

In [ ]:
OPCOES = {"(a) base + traços": 1_639_986, "(b) base + Williams & Best": 1_101_401, "(c) só base": 462_169}
COTA_SEMANAL_H = 30
tempo = tabela[tabela["precisão"] == "fp16"].set_index("modelo")["ms por trecho"]
linhas = []
for opcao, n in OPCOES.items():
    linha = {"opção": opcao, "trechos por modelo": f"{n:,}"}
    for modelo, ms in tempo.items():
        linha[modelo.split("/")[-1] + " (h)"] = round(n * ms / 1000 / 3600, 1) if pd.notna(ms) else None
    linha["total 3 modelos (h)"] = round(n * tempo.sum() / 1000 / 3600, 1)
    linha[f"cabe em {COTA_SEMANAL_H} h?"] = "sim" if linha["total 3 modelos (h)"] <= COTA_SEMANAL_H else "não"
    linhas.append(linha)
projecao = pd.DataFrame(linhas)
projecao

## Resumo para copiar e mandar

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print(tabela.to_string(index=False))
print()
print(projecao.to_string(index=False))